In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium
import json
import requests

# Trabajar con archivo 'Transportes'

In [2]:
path = "data/02-Secretaria/"

In [3]:
tra = pd.read_csv(f"{path}RECORRIDOS_TRANSPORTES_ARANJUEZ_SANTA_CRUZ.csv", encoding="latin-1", sep=";")
tra.columns = tra.columns.str.lower()

In [4]:
# Atributo esarchivolinea, puede ser interesante

columnas_drop = ["empresa_transporte", "idpersona", "esarchivolinea", "ordenregistro",
       "tipocarga", "idpersonausuario", "archivofirmado","fechavencimientofirma", "indfechacreacion", "fecha_carga",
       "ultimoestadopuertas", "presentocambioenpuertas", "ordenregistro","ipmaquina", "codigo_id",
       "id_tipo_ruta", "nombre_ruta", "secuenciarecorrido", "cantidadregistros", "ultimafecharegistro",
       "distanciarecorrido", "min_registros_ruta", "nombre_ruta", "recorridoincumplido", "distanciaincumplidainicio",
       "distanciaincumplidafin", "idempresa", "longitudruta", "tipo_ruta","fechafindespacho", "recorridofinalizado",
       ]

tra = tra.drop(columnas_drop, axis=1)

In [5]:
tra["ultimalatitudregistro"] = pd.to_numeric(tra["ultimalatitudregistro"].str.replace(",", "."))
tra["ultimalongitudregistro"] = pd.to_numeric(tra["ultimalongitudregistro"].str.replace(",", "."))

In [6]:
tra["codigo_ruta"].unique()

array([nan, '024', '041 I ', '042', '023', '022', '041 D', '51-52 CV-O'],
      dtype=object)

In [7]:
example1 = tra.groupby(("codigo_ruta")).get_group(("022")).sort_values(by=["ultimalongitudregistro","ultimalatitudregistro"])
coords1 = [coord[::-1] for coord in example1[["ultimalongitudregistro", "ultimalatitudregistro"]].values.tolist()]

example2 = tra.groupby(("codigo_ruta")).get_group(("042")).sort_values(by=["ultimalongitudregistro","ultimalatitudregistro"])
coords2 = [coord[::-1] for coord in example2[["ultimalongitudregistro", "ultimalatitudregistro"]].values.tolist()]

In [8]:
# Create a map object centered at a specific location
# m = folium.Map(location=coords1[0], zoom_start=13)

# for coord in coords1:
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="blue", icon="info-sign"),
#         tooltip=""
#     ).add_to(m)

# for coord in coords2:
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="red", icon="info-sign"),
#         tooltip=""
#     ).add_to(m)

# m.save("mapa.html")


In [9]:
tra["fechainiciodespacho"]

0        3/27/2026 11:55 PM
1        3/27/2026 11:45 PM
2        3/27/2026 11:30 PM
3        3/27/2026 11:18 PM
4        3/27/2026 11:10 PM
                ...        
10598             3/17/2026
10599    3/16/2026 11:31 PM
10600    3/16/2026 11:15 PM
10601    3/16/2026 11:10 PM
10602    3/16/2026 11:00 PM
Name: fechainiciodespacho, Length: 10603, dtype: object

In [10]:
#Convert format
tra['fechainiciodespacho'] = pd.to_datetime(tra['fechainiciodespacho'], format='%m/%d/%Y %I:%M %p', errors='coerce')
tra['fechainiciodespacho'].dropna(inplace=True)

In [11]:
tra["fechainicio"] = tra.fechainiciodespacho.dt.date
tra["horainicio"] = tra.fechainiciodespacho.dt.hour

In [12]:
print("Varianza según la diferencia de horas de salida por bus, de cada ruta:")
# print(summary.groupby("codigo_ruta").std().nsmallest())

Varianza según la diferencia de horas de salida por bus, de cada ruta:


In [13]:
# Hallar la ruta con menor varianza en la cantidad de buses
ruta_min = tra.groupby(by="codigo_ruta")["idvehiculo"].nunique().nsmallest()

ruta_min

codigo_ruta
51-52 CV-O     1
042           18
022           28
023           29
041 D         33
Name: idvehiculo, dtype: int64

In [14]:
placas = ["STV889","WMQ199","WMQ475","EQT572","EQT960","WMQ211","WMQ800","WMQ124",
"EQT153","EQT626","TSZ911","EQT786","WMQ783","WMQ799","FWK874","EQR887","WMR092",
"EQS970","FWK922","FWL396","EQT038","FWK726","SMV024","EQS782","FWK060","FWL282",
"EQT067","EQS783","EQT189","EQS704","EQW266","EQT152","FVY459","EQT896","EQT800",
"EQT808","EQT809","EQT810","WMQ798","EQT907","EQW005","EQW006","EQW265","EQW289"]

In [15]:
# placas = ["EQS783",
#  "WMP989",
#  "FWK874",
#  "EQT504",
#  "WMR092",
#  "FWK922",
#  "EQT189",
#  "EQT786",
#  "EQT572"]

placas_presentes = tra[tra["placavehiculo"].isin(placas)]["placavehiculo"].unique()

agrupadas = tra.groupby(by="placavehiculo")
for placa in placas_presentes:
    coords = agrupadas.get_group(placa)[["ultimalatitudregistro","ultimalongitudregistro"]].values.tolist()
    print(f"--Bus de placa: {placa}")
    print(agrupadas.get_group(placa)["codigo_ruta"].unique())

#     m = folium.Map(location=coords[0], zoom_start=13)
#     for coord in coords:
#         folium.Marker(
#             location = coord,
#             icon=folium.Icon(color="red", icon="info-sign"),
#             tooltip=""
#         ).add_to(m)

#     m.save("mapa_"+placa+".html")

--Bus de placa: EQT896
[nan '042' '024']
--Bus de placa: FWK922
[nan '022' '041 I ' '041 D']
--Bus de placa: STV889
[nan '023' '024']
--Bus de placa: WMQ475
[nan '023' '024']
--Bus de placa: WMQ783
[nan '023' '024']
--Bus de placa: EQW266
[nan '041 I ' '041 D']
--Bus de placa: FWL282
[nan '041 I ' '041 D' '024' '023' '022']
--Bus de placa: WMQ800
[nan '023']
--Bus de placa: FWK060
[nan '041 I ' '041 D']
--Bus de placa: EQT786
[nan '023']
--Bus de placa: EQR887
[nan '041 I ' '041 D' '024']
--Bus de placa: WMR092
[nan '022' '041 I ' '041 D']
--Bus de placa: WMQ211
[nan '023' '024']
--Bus de placa: WMQ124
[nan '023']
--Bus de placa: EQT572
[nan '023' '024']
--Bus de placa: EQT038
['022' nan '041 I ' '041 D']
--Bus de placa: EQS970
[nan '022' '024' '041 I ' '041 D']
--Bus de placa: WMQ199
[nan '023' '024']
--Bus de placa: EQT152
[nan '041 I ' '041 D' '51-52 CV-O' '024']
--Bus de placa: EQT153
[nan '023']
--Bus de placa: WMQ799
[nan '023']
--Bus de placa: EQS704
[nan '022']
--Bus de placa: 

# Trabajar con archivo 'Pasajeros'

In [16]:
pas = pd.read_csv(f"{path}PASAJEROS_TRANSPORTES_ARANJUEZ_SANTA_CRUZ.csv", encoding="latin-1", sep=";")
pas.columns = pas.columns.str.lower()

In [17]:
columnas_drop = ["empresa"]
pas = pas.drop(columnas_drop, axis=1)

In [18]:
pas

,codigoruta,rutaot,descripcion,fecha,horaregistro,longitud,latitud,velocidad,passuben,pasbajan,nombrearchivo,consecutivo
0,121265,041 D,ARANJUEZ ANILLO,16/03/2026,07:46:43,"-75,552681","6,286985",9,4,0,8909119066EQR88716032026074532,4
1,121265,041 D,ARANJUEZ ANILLO,16/03/2026,07:48:04,"-75,552689","6,285443",11,2,0,8909119066EQR88716032026074532,8
2,121265,041 D,ARANJUEZ ANILLO,16/03/2026,07:49:17,"-75,553589","6,284739",14,2,0,8909119066EQR88716032026074532,11
3,121265,041 D,ARANJUEZ ANILLO,16/03/2026,07:51:38,"-75,554199","6,282160",15,1,0,8909119066EQR88716032026074532,17
4,121265,041 D,ARANJUEZ ANILLO,16/03/2026,07:52:10,"-75,555618","6,282425",20,1,0,8909119066EQR88716032026074532,18
...,...,...,...,...,...,...,...,...,...,...,...,...
246043,121265,022,SANTA CRUZ - TERMINAL,27/03/2026,21:10:50,"-75,570923","6,249527",10,1,0,8909119066WMR09227032026202031,109
246044,121265,022,SANTA CRUZ - TERMINAL,27/03/2026,21:17:56,"-75,568626","6,261690",25,1,0,8909119066WMR09227032026202031,126
246045,121265,022,SANTA CRUZ - TERMINAL,27/03/2026,21:26:32,"-75,563805","6,277751",12,1,0,8909119066WMR09227032026202031,145
246046,121265,022,SANTA CRUZ - TERMINAL,27/03/2026,21:38:07,"-75,555962","6,291877",0,0,1,8909119066WMR09227032026202031,170


In [19]:
# Cast to numeric the lon, lat 
pas["longitud"] = pd.to_numeric(pas["longitud"].str.replace(",", "."))
pas["latitud"] = pd.to_numeric(pas["latitud"].str.replace(",", "."))

In [20]:
# Cast to datetime
pas["fecha"] = pd.to_datetime(pas["fecha"], format='%d/%m/%Y')
pas["horaregistro"] = pd.to_timedelta(pas["horaregistro"])

In [21]:
# This form of filter, and groupping gives too chaotic polylines
pas_rutaot1 = pas.groupby(("rutaot")).get_group(("042"))

pas_rutaot1 = pas_rutaot1[pas_rutaot1["fecha"] == pas_rutaot1["fecha"].iat[0]]

ordenada = pas.sort_values(by=["longitud","latitud"], ascending=False)

coords1 = [coord[::-1] for coord in pas_rutaot1[:1000][["longitud", "latitud"]].values.tolist()]

In [22]:
# Delete point too close
i = 0
while(i < len(coords1)-1):
    if geodesic(coords1[i],coords1[i+1]).meters < 30:
        coords1.pop(i)
    else:
        i += 1

In [23]:
# ---- Filtrar 

pas_rutaot1 = pas.groupby("rutaot").get_group("042")
pas_rutaot1 = pas_rutaot1[pas_rutaot1["fecha"] == pas_rutaot1["fecha"].iat[0]]
pas_rutaot1 = pas_rutaot1.sort_values(by="horaregistro").reset_index(drop=True)

# --- Infer vehicle trips by detecting large jumps ---
MAX_SPEED_KMH = 60  # max realistic bus speed

def compute_trip_id(df, max_speed_kmh=60):
    trip_ids = [0]
    current_trip = 0

    for i in range(1, len(df)):
        prev = df.iloc[i-1]
        curr = df.iloc[i]

        dist_km = geodesic(
            (prev["latitud"], prev["longitud"]),
            (curr["latitud"], curr["longitud"])
        ).km

        time_diff = (curr["horaregistro"] - prev["horaregistro"]).total_seconds() / 3600

        if time_diff > 0:
            speed = dist_km / time_diff
        else:
            speed = float("inf")

        # If speed is physically impossible different vehicle
        if speed > max_speed_kmh:
            current_trip += 1

        trip_ids.append(current_trip)

    df["trip_id"] = trip_ids
    return df

pas_rutaot1 = compute_trip_id(pas_rutaot1)

print(pas_rutaot1["trip_id"].value_counts())

# Pick the longest trip (most GPS pings = most complete route)
best_trip = pas_rutaot1["trip_id"].value_counts().idxmax()
single_trip = pas_rutaot1[pas_rutaot1["trip_id"] == best_trip].reset_index(drop=True)

coords1 = single_trip[["latitud", "longitud"]].values.tolist()

trip_id
0       16
2929     9
2917     7
640      7
131      6
        ..
1090     1
1091     1
1092     1
1093     1
1465     1
Name: count, Length: 2930, dtype: int64


In [24]:
single_trip= single_trip.sort_values(by=["horaregistro"])
coords1 = single_trip[["latitud", "longitud"]].values.tolist()

In [25]:
pas

,codigoruta,rutaot,descripcion,fecha,horaregistro,longitud,latitud,velocidad,passuben,pasbajan,nombrearchivo,consecutivo
0,121265,041 D,ARANJUEZ ANILLO,2026-03-16,0 days 07:46:43,-75.552681,6.286985,9,4,0,8909119066EQR88716032026074532,4
1,121265,041 D,ARANJUEZ ANILLO,2026-03-16,0 days 07:48:04,-75.552689,6.285443,11,2,0,8909119066EQR88716032026074532,8
2,121265,041 D,ARANJUEZ ANILLO,2026-03-16,0 days 07:49:17,-75.553589,6.284739,14,2,0,8909119066EQR88716032026074532,11
3,121265,041 D,ARANJUEZ ANILLO,2026-03-16,0 days 07:51:38,-75.554199,6.282160,15,1,0,8909119066EQR88716032026074532,17
4,121265,041 D,ARANJUEZ ANILLO,2026-03-16,0 days 07:52:10,-75.555618,6.282425,20,1,0,8909119066EQR88716032026074532,18
...,...,...,...,...,...,...,...,...,...,...,...,...
246043,121265,022,SANTA CRUZ - TERMINAL,2026-03-27,0 days 21:10:50,-75.570923,6.249527,10,1,0,8909119066WMR09227032026202031,109
246044,121265,022,SANTA CRUZ - TERMINAL,2026-03-27,0 days 21:17:56,-75.568626,6.261690,25,1,0,8909119066WMR09227032026202031,126
246045,121265,022,SANTA CRUZ - TERMINAL,2026-03-27,0 days 21:26:32,-75.563805,6.277751,12,1,0,8909119066WMR09227032026202031,145
246046,121265,022,SANTA CRUZ - TERMINAL,2026-03-27,0 days 21:38:07,-75.555962,6.291877,0,0,1,8909119066WMR09227032026202031,170


In [26]:
pas_rutaot1 = pas.groupby(("rutaot")).get_group(("041 I "))

ordenada = pas_rutaot1.sort_values(by=["horaregistro"], ascending=False)

coords1 = [coord[::-1] for coord in ordenada[:1000][["longitud", "latitud"]].values.tolist()]

pas_rutaot2 = pas.groupby(("rutaot")).get_group(("041 D"))

ordenada = pas_rutaot2.sort_values(by=["horaregistro"], ascending=False)

coords2 = [coord[::-1] for coord in ordenada[:1000][["longitud", "latitud"]].values.tolist()]

In [27]:
# Create a map object centered at a specific location
# m = folium.Map(location=coords1[0], zoom_start=13)

# for i,coord in enumerate(coords1):
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="blue", icon="info-sign"),
#         tooltip="041 I"
#     ).add_to(m)

# for coord in coords2:
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="red", icon="info-sign"),
#         tooltip="041 D"
#     ).add_to(m)

# m.save("mapa.html")

In [28]:
tra["codigo_ruta"].value_counts()

codigo_ruta
024           2215
023           1384
022           1008
041 I          891
042            853
041 D          576
51-52 CV-O       1
Name: count, dtype: int64

In [29]:
# with open ("data/ruta_coords.json","w") as f:
#     json.dump({"coords":coords1}, f, indent=1)

In [30]:
# def get_elevations_batch(coords):
#     """coords: list of (lat, lon) tuples"""
#     locations = "|".join([f"{lat},{lon}" for lat, lon in coords])
#     url = f"https://api.open-elevation.com/api/v1/lookup?locations={locations}"
#     response = requests.get(url).json()
#     return [r["elevation"] for r in response["results"]]

# # Apply to the dataframe
# coords = list(zip(single_trip["latitud"], single_trip["longitud"]))
# single_trip["elevation"] = get_elevations_batch(coords)

# # Compute inclination between consecutive points
# single_trip["dist_m"] = [0] + [
#     geodesic(coords[i-1], coords[i]).meters 
#     for i in range(1, len(coords))
# ]

# single_trip["inclination_deg"] = np.degrees(
#     np.arctan2(
#         single_trip["elevation"].diff(),
#         single_trip["dist_m"]
#     )
# )

# Merge de los datasets limpios

In [31]:
trans_pas = tra.merge(pas, on="nombrearchivo", how="inner")

In [32]:
trans_pas[trans_pas["nombrearchivo"] =="8909119066EQW26627032026213532"]

,nombrearchivo,idvehiculo,placavehiculo,idruta,codigo_ruta,fechainiciodespacho,fecha_creacion,cantpasajerossuben,cantpasajerosbajan,ultimalongitudregistro,...,rutaot,descripcion,fecha,horaregistro,longitud,latitud,velocidad,passuben,pasbajan,consecutivo
1,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:11:09,-75.565483,6.247818,0,1,0,77
2,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:12:35,-75.565483,6.247818,0,1,0,80
3,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:13:06,-75.565483,6.247818,0,2,0,81
4,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:13:45,-75.565483,6.247818,0,1,0,83
5,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:42:09,-75.550041,6.284740,14,0,2,142
6,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:42:35,-75.550003,6.285378,13,2,0,144
7,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:43:06,-75.549934,6.286220,0,0,1,145
8,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:43:24,-75.549896,6.286541,14,0,1,146
9,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:43:55,-75.549164,6.286475,14,0,1,147
10,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:45:07,-75.546883,6.286173,9,0,2,151


In [33]:
trans_pas.codigo_ruta.unique()

array(['024', '041 I ', '042', '023', '022', '041 D', '51-52 CV-O'],
      dtype=object)

In [34]:
ruta_041_enriquecida = trans_pas[trans_pas["codigo_ruta"].isin(["041 I ", "041 D"])]
ruta_041_enriquecida.to_csv("data/rich_041.csv",index=False)

In [37]:
trans_pas.iloc[0]["placavehiculo"]="placa123"

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3016\3721778070.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trans_pas.iloc[0]["placavehiculo"]="placa123"


In [40]:
x = trans_pas.iloc[0]
x["placavehiculo"] = "placa123"

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3016\4203180618.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x["placavehiculo"] = "placa123"


In [43]:
trans_pas.iloc[0] = x

In [46]:
trans_pas.iloc[0:]

,nombrearchivo,idvehiculo,placavehiculo,idruta,codigo_ruta,fechainiciodespacho,fecha_creacion,cantpasajerossuben,cantpasajerosbajan,ultimalongitudregistro,...,rutaot,descripcion,fecha,horaregistro,longitud,latitud,velocidad,passuben,pasbajan,consecutivo
0,8909119066TSK92127032026220539,6375,placa123,803,024,2026-03-27 22:05:00,27/03/2026 22:05:39,1,0,-75.564110,...,024,024-BARRIAL,2026-03-27,0 days 22:11:45,-75.556931,6.300485,0,1,0,16
1,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:11:09,-75.565483,6.247818,0,1,0,77
2,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:12:35,-75.565483,6.247818,0,1,0,80
3,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:13:06,-75.565483,6.247818,0,2,0,81
4,8909119066EQW26627032026213532,6522,EQW266,530,041 I,2026-03-27 21:35:00,27/03/2026 21:35:32,7,7,-75.546364,...,041 I,ARANJUEZ GUADALUPE,2026-03-27,0 days 22:13:45,-75.565483,6.247818,0,1,0,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222621,8909119066TPV87117032026034533,5258,TPV871,525,022,2026-03-17 03:45:00,17/03/2026 03:45:33,11,11,-75.547646,...,022,SANTA CRUZ - TERMINAL,2026-03-17,0 days 04:13:39,-75.567451,6.254076,12,0,2,70
222622,8909119066TPV87117032026034533,5258,TPV871,525,022,2026-03-17 03:45:00,17/03/2026 03:45:33,11,11,-75.547646,...,022,SANTA CRUZ - TERMINAL,2026-03-17,0 days 04:15:47,-75.570534,6.252877,14,1,0,76
222623,8909119066TPV87117032026034533,5258,TPV871,525,022,2026-03-17 03:45:00,17/03/2026 03:45:33,11,11,-75.547646,...,022,SANTA CRUZ - TERMINAL,2026-03-17,0 days 04:17:47,-75.572540,6.248426,0,0,1,80
222624,8909119066TPV87117032026034533,5258,TPV871,525,022,2026-03-17 03:45:00,17/03/2026 03:45:33,11,11,-75.547646,...,022,SANTA CRUZ - TERMINAL,2026-03-17,0 days 04:18:48,-75.572411,6.246685,29,0,3,84
